In [1]:


import pandas as pd
from datasets import load_dataset

!pip install datasets

#load the ag_news from hugging face
dataset = load_dataset("wangrongsheng/ag_news")

#convert the training split to a pandas Dataframe for easy preprocessing
df=pd.DataFrame(dataset['train'])
df.head()

README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 18.6MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.23MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

,text,label
0,Wall St. Bears Claw Back Into the Black (Reute...,2
1,Carlyle Looks Toward Commercial Aerospace (Reu...,2
2,Oil and Economy Cloud Stocks' Outlook (Reuters...,2
3,Iraq Halts Oil Exports from Main Southern Pipe...,2
4,"Oil prices soar to all-time record, posing new...",2


In [2]:
train_df = pd.DataFrame(dataset["train"])
test_df = pd.DataFrame(dataset["test"])

In [3]:
import re

In [4]:
def clean_text(text):
    # Convert text to lowercase
    text = text.lower()

    # Remove punctuation, numbers, and special characters
    text = re.sub(r'[^a-z\s]', '', text)

    return text

In [5]:
train_df["text"] = train_df["text"].apply(clean_text)
test_df["text"] = test_df["text"].apply(clean_text)

In [6]:
train_df.head()

,text,label
0,wall st bears claw back into the black reuters...,2
1,carlyle looks toward commercial aerospace reut...,2
2,oil and economy cloud stocks outlook reuters r...,2
3,iraq halts oil exports from main southern pipe...,2
4,oil prices soar to alltime record posing new m...,2


In [7]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [8]:
#This creates a tokenizer object that will build a vocabulary.
tokenizer = Tokenizer()

In [9]:
#This scans all the training articles and assigns a unique integer ID to each unique word.
tokenizer.fit_on_texts(train_df["text"])

In [10]:
#Each article is converted from words into a sequence of numbers.
X_train = tokenizer.texts_to_sequences(train_df["text"])
X_test = tokenizer.texts_to_sequences(test_df["text"])

In [11]:
#Print the first 10 words in the vocabulary
print(list(tokenizer.word_index.items())[:10])

[('the', 1), ('to', 2), ('a', 3), ('of', 4), ('in', 5), ('and', 6), ('on', 7), ('for', 8), ('s', 9), ('that', 10)]


In [12]:
#To View the Tokenized Text
print(train_df["text"][0])
print(X_train[0])

wall st bears claw back into the black reuters reuters  shortsellers wall streets dwindlingband of ultracynics are seeing green again
[391, 324, 1525, 14260, 99, 54, 1, 812, 23, 23, 38863, 391, 1988, 50537, 4, 38864, 34, 3893, 737, 295]


In [13]:
#Import pad_sequences
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [14]:
#Set the Maximum Sequence Length
max_length = 50

In [15]:
#Pad and Truncate the Training Data
X_train = pad_sequences(
    X_train,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

# Pad and Truncate the Testing Data
X_test = pad_sequences(
    X_test,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

In [16]:
#Check the Shape
print("Training Data Shape:", X_train.shape)
print("Testing Data Shape:", X_test.shape)

Training Data Shape: (120000, 50)
Testing Data Shape: (7600, 50)


In [17]:
#View a Padded Sequence
print(X_train[0])

[  391   324  1525 14260    99    54     1   812    23    23 38863   391
  1988 50537     4 38864    34  3893   737   295     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0]


In [18]:
from tensorflow.keras.utils import to_categorical

In [19]:
y_train = to_categorical(train_df["label"])
y_test = to_categorical(test_df["label"])

In [20]:
print(train_df["label"].head())

0    2
1    2
2    2
3    2
4    2
Name: label, dtype: int64


In [21]:
print(y_train[:5])

[[0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]]


In [22]:
#Check the Shape
print("Training Labels Shape:", y_train.shape)
print("Testing Labels Shape:", y_test.shape)

Training Labels Shape: (120000, 4)
Testing Labels Shape: (7600, 4)


In [23]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

In [24]:
vocab_size = len(tokenizer.word_index) + 1
print("Vocabulary Size:", vocab_size)

Vocabulary Size: 91344


In [25]:
model = Sequential()

# Embedding Layer
model.add(Embedding(input_dim=vocab_size,
                    output_dim=64,
                    input_length=max_length))

# SimpleRNN Layer
model.add(SimpleRNN(64))

# Output Layer
model.add(Dense(4, activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [26]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [27]:
# Compile the RNN model
model.compile(
    optimizer='adam',                  # Optimizer that updates model weights
    loss='categorical_crossentropy',   # Loss function for multi-class classification
    metrics=['accuracy']               # Evaluate the model using accuracy
)

In [28]:
# Train the RNN model
history = model.fit(
    X_train,                # Training input data
    y_train,                # One-hot encoded training labels
    epochs=5,               # Number of times the model sees the entire training dataset
    batch_size=64,          # Number of samples processed before updating the weights
    validation_split=0.2    # Use 20% of the training data for validation
)

Epoch 1/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 111s 72ms/step - accuracy: 0.8350 - loss: 0.4884 - val_accuracy: 0.8648 - val_loss: 0.4004
Epoch 2/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 145s 75ms/step - accuracy: 0.9220 - loss: 0.2556 - val_accuracy: 0.8686 - val_loss: 0.4098
Epoch 3/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 109s 73ms/step - accuracy: 0.9416 - loss: 0.1917 - val_accuracy: 0.8698 - val_loss: 0.4208
Epoch 4/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 109s 73ms/step - accuracy: 0.9538 - loss: 0.1513 - val_accuracy: 0.8783 - val_loss: 0.4176
Epoch 5/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 141s 72ms/step - accuracy: 0.9632 - loss: 0.1217 - val_accuracy: 0.8685 - val_loss: 0.4654


In [29]:
model.compile(
    optimizer='adam',                  # Optimizer used to update weights
    loss='categorical_crossentropy',   # Loss function for multi-class classification
    metrics=['accuracy']               # Display accuracy during training
)
# Train the RNN Model

history = model.fit(
    X_train,                # Input training data
    y_train,                # Target labels
    epochs=5,               # Number of training iterations
    batch_size=64,          # Samples processed in each batch
    validation_split=0.2    # Reserve 20% of training data for validation
)

Epoch 1/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 116s 76ms/step - accuracy: 0.9688 - loss: 0.1039 - val_accuracy: 0.8725 - val_loss: 0.4601
Epoch 2/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 141s 75ms/step - accuracy: 0.9763 - loss: 0.0804 - val_accuracy: 0.8649 - val_loss: 0.5266
Epoch 3/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 113s 76ms/step - accuracy: 0.9687 - loss: 0.1053 - val_accuracy: 0.8568 - val_loss: 0.5208
Epoch 4/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 117s 78ms/step - accuracy: 0.9801 - loss: 0.0648 - val_accuracy: 0.8602 - val_loss: 0.5726
Epoch 5/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 115s 76ms/step - accuracy: 0.9874 - loss: 0.0440 - val_accuracy: 0.8574 - val_loss: 0.6327


In [30]:
# Evaluate the trained model on the test dataset
loss, accuracy = model.evaluate(
    X_test,     # Test input data
    y_test      # Test labels
)

# Display the evaluation results
print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

238/238 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8732 - loss: 0.5774
Test Loss: 0.577360212802887
Test Accuracy: 0.8731579184532166


In [31]:
# Display the training accuracy from the last epoch
print("Training Accuracy:", history.history['accuracy'][-1])

# Display the validation accuracy from the last epoch
print("Validation Accuracy:", history.history['val_accuracy'][-1])

Training Accuracy: 0.9873645901679993
Validation Accuracy: 0.8574166893959045


In [32]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (64, 50, 64)           │     5,846,016 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (64, 64)               │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (64, 4)                │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 17,563,598 (67.00 MB)

 Trainable params: 5,854,532 (22.33 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 11,709,066 (44.67 MB)